In [8]:
# We use the composite chained classifier-regressor model in 4B_5 to predict daily rasters 

# 1. Load all predictor rasters for the day
# 2. Stack and create a df
# 3. Format df to fit column requirement of the model
# 4. Make predictions in the df of precipitation_sum
# 5. Make a raster based on the predicted precipitation
# 4B_6_2 predict_rasters.ipynb --> Saved the day's predicted precipitation raster

In [9]:
import os
import gc
import glob
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

start_year, start_month, start_day = 2001, 1, 1
end_year,   end_month, end_day   = 2005, 12, 1

# paths for static rasters
static_paths = {
    "cluster":         r"..\Data\Processed\Spatial_clustering\Precipitation_clusters.tif",
    "elev_mean":       r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_mean.tif",
    "elev_std":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_std.tif",
    "elev_max":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif",
    "elev_min":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_min.tif",
    "slope_mean":      r"..\Data\Datasets\DEM_1km\DEM_1km_slope_mean.tif",
    "aspect_mean":     r"..\Data\Datasets\DEM_1km\DEM_1km_aspect_mean.tif",
    "elev_smoothed":   r"..\Data\Processed\Fetch\Preprocessing\dem_smoothed.tif",
    "fetch_1000m_dist":r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_1000m.tif",
    "fetch_1000m_dir": r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_1000m.tif",
    "fetch_500m_dist": r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_500m.tif",
    "fetch_500m_dir":  r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_500m.tif",
    "local_relief":    r"..\Data\Processed\Fetch\Preprocessing\local_relief.tif",
}

# template grid
template_path = r"..\Data\Datasets\CHIRPS_prec\CHIRPS_prec_19910101.tif"
template = rxr.open_rasterio(template_path)

out_base = r"..\Data\Processed\Bias_correction_daily\Predicted_rasters"
temp_base = os.path.join(out_base, "temp")
daily_dynamic_pred_dir = os.path.join(temp_base, "daily_dynamic_pred_df")
monthly_parquet_dir = os.path.join(temp_base, "monthly_parquet_df") # to store the daily predictions in parquet_dir
static_dir = os.path.join(temp_base, "static_df")

for d in [temp_base, daily_dynamic_pred_dir, monthly_parquet_dir, static_dir]:
    if not os.path.exists(d):
        os.makedirs(d)

In [10]:
# HELPERS

def doy_cyclical(year: int, doy: int):
    """Return (sin, cos) encoding for the day of the year (1-365)"""
    if (year % 4) == 0:
        angle = 2 * np.pi * doy / 366
    else:
        angle = 2 * np.pi * doy / 366
    return np.sin(angle), np.cos(angle)

def day_label(year: int, month: int, day: int) -> str:
    return f"{year}{month:02d}{day:02d}"

def build_daily_paths(year: int, month: int, day: int) -> dict:
    return {
        "era5l": fr"..\Data\Datasets\ERA5L_prec\ERA5L_prec_{year}{month:02d}{day:02d}.tif",
        "chirps": fr"..\Data\Datasets\CHIRPS_prec\CHIRPS_prec_{year}{month:02d}{day:02d}.tif",
        "har": fr"..\Data\Datasets\HAR_prec\HAR_prec_{year}{month:02d}{day:02d}.tif",
        "persiann_prec":  fr"..\Data\Datasets\PERSIANN_prec\PERSIANN_precTot_{year}{month:02d}{day:02d}.tif",
        "mswep_prec":     fr"..\Data\Datasets\MSWEP_prec\MSWEP_Prec_{year}{month:02d}{day:02d}.tif",
        "gldas_air_temp": fr"..\Data\Datasets\GLDAS\air_temp\GLDAS_air_temp_{year}{month:02d}{day:02d}.tif",
        "gldas_lwdown": fr"..\Data\Datasets\GLDAS\lwdown\GLDAS_lw_down_{year}{month:02d}{day:02d}.tif",
        "gldas_specific_humidity": fr"..\Data\Datasets\GLDAS\specific_humidity\GLDAS_specific_humidity_{year}{month:02d}{day:02d}.tif",
        "gldas_surface_pressure": fr"..\Data\Datasets\GLDAS\surface_pressure\GLDAS_surface_pressure_{year}{month:02d}{day:02d}.tif",
        "gldas_swdown": fr"..\Data\Datasets\GLDAS\swdown\GLDAS_sw_down_{year}{month:02d}{day:02d}.tif",
        "gldas_wind_speed": fr"..\Data\Datasets\GLDAS\wind_speed\GLDAS_wind_speed_{year}{month:02d}{day:02d}.tif",
        "insitu_kriged": fr"..\Data\Processed\Bias_correction_daily\Insitu_kriged_daily\insitu_precip_{year}{month:02d}{day:02d}.tif"
    }

In [11]:
# 1. Static feature dataframe -------------------------------------

static_csv = os.path.join(static_dir, "static_df.csv")
 
if os.path.exists(static_csv):
    print("Loading cached static df...")
    static_df = pd.read_csv(static_csv)
else:
    print("Building static df from rasters...")
    aligned = []
    for name, path in static_paths.items():
        da = rxr.open_rasterio(path, masked=True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template).squeeze("band", drop=True)
        da.name = name
        if "spatial_ref" in da.coords:
            da = da.drop_vars("spatial_ref")
        aligned.append(da)
    static_xr = xr.merge(aligned, compat="override")
    static_df  = static_xr.to_dataframe().reset_index()
    static_df.to_csv(static_csv, index=False)
    del aligned, static_xr
    gc.collect()
 
print(f"Static df: {len(static_df):,} pixels")

Building static df from rasters...
Static df: 15,753 pixels


In [12]:
# 2. Main loop daily to build the daily dynamic df -------------------------------------

daily_range = pd.period_range(
    start = f"{start_year}-{start_month:02d}-{start_day:02d}",
    end = f"{end_year}-{end_month:02d}-{end_day:02d}",
    freq= "D"
)

print(f"Processing {len(daily_range)} days")

for period in tqdm(daily_range, desc = "Daily feature extraction"):
    year = period.year
    month = period.month
    day = period.day
    label = day_label(year, month, day)

    pred_out = os.path.join(daily_dynamic_pred_dir, f"dynamic_df_{label}.parquet")

    if os.path.exists(pred_out):
        continue 

    paths = build_daily_paths(year, month, day)
    aligned = []
    missing_vars = []

    for name, path in paths.items():
        if not os.path.exists(path):
            missing_vars.append(name)
            continue
        da = rxr.open_rasterio(path, masked=True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template).squeeze("band", drop=True)
        da.name = name
        if "spatial_ref" in da.coords:
            da = da.drop_vars("spatial_ref")
        aligned.append(da)
        da.close()
    
    if missing_vars:
        tqdm.write(f"Warning: Missing rasters for {label}: {', '.join(missing_vars)}")

    if not aligned:
        tqdm.write(f"Warning: No rasters found for {label}, skipping.")
        continue

    dynamic_xr = xr.merge(aligned, compat="override")
    dynamic_df = dynamic_xr.to_dataframe().reset_index()

    for name in missing_vars:
        dynamic_df[name] = np.nan

    del aligned, dynamic_xr
    gc.collect()

    daily_grid = pd.merge(static_df, dynamic_df, on=["x", "y"], how="inner")

    sin_doy, cos_doy = doy_cyclical(year, period.dayofyear)
    daily_grid["doy_sin"] = sin_doy
    daily_grid["doy_cos"] = cos_doy
    daily_grid["year"] = year
    daily_grid["month"] = month
    daily_grid["day"] = day

    daily_grid.to_parquet(pred_out, index=False)
    del daily_grid
    gc.collect()

Processing 1796 days


Daily feature extraction:   0%|          | 0/1796 [00:00<?, ?it/s]

Daily feature extraction:  98%|█████████▊| 1764/1796 [13:51<00:13,  2.43it/s]

Daily feature extraction: 100%|██████████| 1796/1796 [14:02<00:00,  2.13it/s]


In [13]:
# 3. Aggregate daily dfs to yearly parquet to final combined df -------------------------------------

def aggregate_to_parquet(daily_dir: str, parquet_dir: str, split_label: str):
    parquet_paths = []

    for year in range(start_year, end_year + 1):
        yearly_path = os.path.join(parquet_dir, f"full_{year}.parquet")
        daily_periods = pd.period_range(
            start = f"{year}-01-01", end=f"{year}-12-31", freq="D"
        )
        expected = [
            os.path.join(daily_dir, f"dynamic_df_{day_label(p.year, p.month, p.day)}.parquet") 
            for p in daily_periods
        ]    
        found = [f for f in expected if os.path.exists(f)]

        if not found:
            print(f"Warning: No daily CSVs found for {year}, skipping.")
        
        if found:
            yearly_df = pd.concat([pd.read_parquet(f) for f in found], ignore_index=True)
            yearly_df.to_parquet(yearly_path, index=False)
            parquet_paths.append(yearly_path)
            print(f"[{split_label}] Year {year}: {len(yearly_df):,} rows saved to {yearly_path}")

    if not parquet_paths:
        print(f"Warning: No yearly parquet files created for {split_label}.")
        return

    final_df = pd.concat([pd.read_parquet(p) for p in parquet_paths], ignore_index=True)
    final_path = os.path.join(out_base, f"full_daily_df.parquet")
    final_df.to_parquet(final_path, index=False)

    final_df.head(10_000).to_csv(
        os.path.join(out_base, f"full_{split_label}_preview.csv"), index=False
    )

    print(f"\\n  [{split_label}] Final df: {len(final_df):,} rows × {len(final_df.columns)} cols")
    print(f"  Saved: {final_path}")
    return final_path

pred_out = aggregate_to_parquet(daily_dynamic_pred_dir, monthly_parquet_dir, "daily_dynamic")

[daily_dynamic] Year 2001: 5,749,845 rows saved to ..\Data\Processed\Bias_correction_daily\Predicted_rasters\temp\monthly_parquet_df\full_2001.parquet
[daily_dynamic] Year 2002: 5,749,845 rows saved to ..\Data\Processed\Bias_correction_daily\Predicted_rasters\temp\monthly_parquet_df\full_2002.parquet
[daily_dynamic] Year 2003: 5,749,845 rows saved to ..\Data\Processed\Bias_correction_daily\Predicted_rasters\temp\monthly_parquet_df\full_2003.parquet
[daily_dynamic] Year 2004: 5,765,598 rows saved to ..\Data\Processed\Bias_correction_daily\Predicted_rasters\temp\monthly_parquet_df\full_2004.parquet
[daily_dynamic] Year 2005: 5,277,255 rows saved to ..\Data\Processed\Bias_correction_daily\Predicted_rasters\temp\monthly_parquet_df\full_2005.parquet
\n  [daily_dynamic] Final df: 28,292,388 rows × 32 cols
  Saved: ..\Data\Processed\Bias_correction_daily\Predicted_rasters\full_daily_df.parquet


In [14]:
import shutil

print("\n" + "=" * 60)
print("Cleaning up intermediate files and directories...")
print("=" * 60)

dirs_to_remove = [
    daily_dynamic_pred_dir,
    monthly_parquet_dir,
    static_dir
]

for d in dirs_to_remove:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Removed directory: {d}")


Cleaning up intermediate files and directories...
Removed directory: ..\Data\Processed\Bias_correction_daily\Predicted_rasters\temp\daily_dynamic_pred_df
Removed directory: ..\Data\Processed\Bias_correction_daily\Predicted_rasters\temp\monthly_parquet_df
Removed directory: ..\Data\Processed\Bias_correction_daily\Predicted_rasters\temp\static_df
